# Mini Project 1 -- How Film Dialogue Reflects the Human-Technology Relationship

**Name:** Riti Upadhyay  
**Dataset:** Cornell Movie Dialogs Corpus  
**Course:** HCDE 530 -- Computational Concepts in HCDE  
**Date:** May 2026

---

## Section 1 -- Overview

### What is this dataset?

The Cornell Movie Dialogs Corpus contains 220,579 conversational exchanges from 617 movie scripts. Each line of dialogue is tagged with the film title, release year, and genre. The data covers films released between the 1930s and 2010s. It was assembled by researchers at Cornell University and is publicly available at https://convokit.cornell.edu/documentation/movie.html.

Film dialogue is a unique record of how ordinary people talked in everyday situations across eight decades. Unlike surveys or interviews, it was not produced for research. It reflects the language of its time without self-consciousness.

### What questions am I trying to answer?

1. Which technology terms spread across the most films per decade, and how did that change over time?
2. Did the emotional language around specific technology terms shift from positive to negative over the decades?
3. When did technology stop being something characters use and start being something that acts on its own?

### Why do these questions matter?

HCI as a field designs products based on assumptions about how people relate to technology. But those assumptions have a cultural history that predates the field. Film dialogue shows what people already expected from technology before anyone designed for those expectations. If characters in 1960s films talk about computers with curiosity, and characters in 2000s films talk about them with anxiety, that shift happened in culture before it showed up in design research. Understanding that arc is useful for anyone designing human-centered systems today.

### How the questions evolved

The original MP1a questions proposed full sentiment scoring and grammatical dependency parsing. Both require NLP tools beyond what this course covers. After feedback, Q2 was reframed as a positive-to-negative emotion word ratio using a curated lexicon (with a volume assistive chart). Q3 was reframed as acting-verb vs tool-verb rates near tech terms in a 6-word window—not dependency parsing. Q1 uses distinct films per term per decade with stacked **share %** so multi-term films are not double-counted. See Section 5 for the full process.

In [15]:
# Setup -- run this cell first
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from pathlib import Path

print("Setup complete.")

Setup complete.


---

## Section 2 -- Data Profile

We load three CSV files produced by `fetch_cornell.py`. That script downloads the Cornell corpus zip directly from Cornell's server, parses the custom `+++$+++` separator format, joins dialogue lines with movie metadata, and saves the results. No manual file handling is needed.

- `cornell_clean.csv` -- one row per dialogue line, full cleaned dataset
- `cornell_tidy.csv` -- tidy format, one row per technology term mention per line
- `cornell_term_by_decade.csv` -- summary, term frequency normalized per 1000 lines per decade

In [16]:
df = pd.read_csv("cornell_clean.csv")
tidy_df = pd.read_csv("cornell_tidy.csv")
term_df = pd.read_csv("cornell_term_by_decade.csv")

print("cornell_clean.csv:", df.shape)
print("cornell_tidy.csv:", tidy_df.shape)
print("cornell_term_by_decade.csv:", term_df.shape)
df.head()

cornell_clean.csv: (304160, 11)
cornell_tidy.csv: (2712, 8)
cornell_term_by_decade.csv: (103, 6)


,movie_id,title,year,decade,genres,text,has_tech,tech_terms_found,first_tech_position,first_tech_category,line_length
0,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",They do not!,False,NaN,NaN,NaN,3
1,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",They do to!,False,NaN,NaN,NaN,3
2,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",I hope so.,False,NaN,NaN,NaN,3
3,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",She okay?,False,NaN,NaN,NaN,2
4,m0,10 things i hate about you,1999,1990,"['comedy', 'romance']",Let's go.,False,NaN,NaN,NaN,3


In [17]:
# df.info() shows column types and how many non-null values each column has.
# first_tech_position and first_tech_category are null for lines with no tech terms -- expected.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 304160 entries, 0 to 304159
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   movie_id             304160 non-null  str    
 1   title                304160 non-null  str    
 2   year                 304160 non-null  int64  
 3   decade               304160 non-null  int64  
 4   genres               304160 non-null  str    
 5   text                 304160 non-null  str    
 6   has_tech             304160 non-null  bool   
 7   tech_terms_found     2624 non-null    str    
 8   first_tech_position  2624 non-null    float64
 9   first_tech_category  2624 non-null    str    
 10  line_length          304160 non-null  int64  
dtypes: bool(1), float64(1), int64(3), str(6)
memory usage: 23.5 MB


In [18]:
# df.describe() summarizes numeric columns.
# line_length averages around 10 words, which is typical for film dialogue.
# first_tech_position ranges from 0 to 1 -- 0 means the term is at the very start of the line.
df.describe()

,year,decade,first_tech_position,line_length
count,304160.000000,304160.000000,2624.000000,304160.000000
mean,1988.164831,1983.495759,0.527040,11.210060
std,17.046154,17.101330,0.270402,12.632913
min,1931.000000,1930.000000,0.000000,0.000000
25%,1984.000000,1980.000000,0.300000,4.000000
50%,1995.000000,1990.000000,0.545455,7.000000
75%,1999.000000,1990.000000,0.763904,14.000000
max,2010.000000,2010.000000,0.989655,582.000000


In [19]:
# isnull().sum() counts missing values per column.
# first_tech_position and first_tech_category are null for lines without tech terms.
# This is expected and those rows are simply excluded from Q3 analysis.
df.isnull().sum()

movie_id                    0
title                       0
year                        0
decade                      0
genres                      0
text                        0
has_tech                    0
tech_terms_found       301536
first_tech_position    301536
first_tech_category    301536
line_length                 0
dtype: int64

**Data profile notes:** The full dataset has 304,160 dialogue lines across 617 films from the 1930s through 2010s. The key columns for this analysis are `decade`, `has_tech`, `tech_terms_found`, `title`, and `text`. The tidy dataset restructures `tech_terms_found` so each term gets its own row -- one observation per row, one variable per column, following the tidy data principle from class. The summary dataset normalizes term counts per 1000 lines per decade so decades with more films in the corpus do not dominate the results.

---

## Section 3 -- Analysis

### Shared definitions

The lexicons below are used across Q2 and Q3. They are defined once here following the professor's guidance that a shared function or definition does not need to be duplicated across question blocks.

In [20]:
# Technology terms tracked across all three questions.
COMM_TERMS = ["phone", "telephone", "radio", "television", "tv", "call", "signal"]
COMP_TERMS = ["computer", "internet", "online", "email", "app", "algorithm",
              "ai", "robot", "digital", "machine", "screen", "device", "network", "data"]
ALL_TERMS = COMM_TERMS + COMP_TERMS

# Positive emotion words -- chosen for low context ambiguity.
POSITIVE = set([
    "love","loved","loving","hope","hoped","hopeful","trust","trusted",
    "joy","joyful","happy","happiness","wonderful","amazing","excellent",
    "brilliant","fantastic","incredible","beautiful","perfect","proud",
    "pride","kind","kindness","gentle","warm","warmth","safe","safety",
    "free","freedom","fun","glad","grateful","gratitude","delight",
    "delighted","cheerful","peaceful","calm","comfort","comfortable",
    "excited","excitement","pleasure","pleased","satisfying","satisfied",
    "confident","confidence","inspired","inspiration","admire","admired",
    "celebrate","celebration","blessed","lucky","fortunate","generous",
    "enthusiastic","enthusiasm","caring","supportive","uplifting",
    "encouraging","relief","relieved","thrilled","content","appreciated",
    "valuable","meaningful","powerful","resilient","brave","courage",
    "courageous","honest","sincere","passionate","creative","wise",
    "clever","helpful","healing","healed","stronger","succeed","success",
    "winning","winner","achievement","accomplished","flourish","thrive",
    "connected","belonging","together","unity","bond","friendship",
    "compassion","like","liked","likes","nice","good","better","best",
    "great","sweet","dear","charming","positive","optimistic","trust",
    "reassured","secure","energized","motivated","valued"
])

# Negative emotion words -- chosen for low context ambiguity.
NEGATIVE = set([
    "fear","feared","fearing","hate","hated","hating","hatred",
    "worry","worried","worrying","danger","dangerous","terrible",
    "horrible","awful","dreadful","frightening","frightened","scared",
    "scary","terrified","terror","panic","panicked","miserable",
    "misery","despair","hopeless","helpless","powerless","worthless",
    "useless","failure","failed","failing","defeat","defeated",
    "destroyed","ruin","ruined","catastrophe","catastrophic",
    "disaster","devastating","devastated","suffering","suffer",
    "pain","painful","agony","agonizing","depressed","depression",
    "anxiety","anxious","stressed","stress","overwhelmed","overwhelm",
    "exhausted","exhaustion","trapped","stuck","isolated","isolation",
    "lonely","loneliness","abandoned","rejection","rejected",
    "betrayal","betrayed","shame","ashamed","guilt","guilty",
    "regret","regretful","angry","anger","rage","furious","fury",
    "bitter","bitterness","resentful","resentment","jealous",
    "jealousy","envy","cruel","cruelty","violent","violence",
    "threat","threatening","hostile","hostility","corrupt",
    "corruption","evil","wicked","sinister","manipulative",
    "manipulation","deception","deceived","lying","lied",
    "confused","confusion","meaningless","empty","emptiness",
    "numb","collapse","collapsed","nightmare","nightmarish",
    "unbearable","intolerable","humiliated","humiliation",
    "disgusting","disgust","repulsed","bad","worst","wrong",
    "broken","hurt","hurting","sad","sadness","unhappy",
    "mad","crazy","stupid","mean","alone","sorry","lost",
    "negative","pessimistic","distrustful","suspicious",
    "insecure","unstable","chaotic","doomed","dead","dying",
    "kill","killed","killing","murder","murderous","deadly"
])

# Acting verbs -- technology as subject performing an action.
ACTING_VERBS = set([
    "decided","decides","decide","chose","chooses","choose",
    "told","tells","tell","said","says","warned","warns",
    "predicted","predicts","predict","learned","learns","learn",
    "knew","knows","know","found","finds","find","showed","shows","show",
    "detected","detects","detect","recognized","recognizes","recognize",
    "identified","identifies","identify","suggested","suggests","suggest",
    "recommended","recommends","recommend","controlled","controls","control",
    "commanded","commands","command","directed","directs","direct",
    "guided","guides","guide","responded","responds","respond",
    "answered","answers","answer","understood","understands","understand",
    "calculated","calculates","calculate","computed","computes","compute",
    "processed","processes","process","generated","generates","generate",
    "created","creates","create","produced","produces","produce",
    "displayed","displays","display","recorded","records","record",
    "stored","stores","store","remembered","remembers","remember",
    "communicated","communicates","communicate","connected","connects","connect",
    "transmitted","transmits","transmit","sent","sends","send",
    "monitored","monitors","monitor","tracked","tracks","track",
    "analyzed","analyzes","analyze","measured","measures","measure",
    "evaluated","evaluates","evaluate","determined","determines","determine",
    "selected","selects","select","rejected","rejects","reject",
    "approved","approves","approve","blocked","blocks","block",
    "allowed","allows","allow","prevented","prevents","prevent",
    "enabled","enables","enable","disabled","disables","disable",
    "triggered","triggers","trigger","activated","activates","activate",
    "initiated","initiates","initiate","terminated","terminates","terminate",
    "reported","reports","report","notified","notifies","notify",
    "alerted","alerts","alert","updated","updates","update",
    "adapted","adapts","adapt","optimized","optimizes","optimize",
    "improved","improves","improve","solved","solves","solve",
    "interpreted","interprets","interpret","translated","translates","translate",
    "searched","searches","search","retrieved","retrieves","retrieve",
    "replaced","replaces","replace","managed","manages","manage",
    "collected","collects","collect","gathered","gathers","gather",
    "simulated","simulates","simulate","calculated","estimated","estimates","estimate"
])

# Tool verbs -- someone acts on technology.
TOOL_VERBS = set([
    "used","use","using","fixed","fix","fixing","bought","buy","buying",
    "built","build","building","installed","install","installing",
    "repaired","repair","repairing","replaced","replace","replacing",
    "upgraded","upgrade","upgrading","pressed","press","pressing",
    "clicked","click","clicking","typed","type","typing",
    "switched","switch","switching","held","hold","holding",
    "turned","turn","turning","plugged","plug","plugging",
    "unplugged","unplug","unplugging","checked","check","checking",
    "operated","operate","operating","adjusted","adjust","adjusting",
    "configured","configure","configuring","programmed","program","programming",
    "deleted","delete","deleting","downloaded","download","downloading",
    "uploaded","upload","uploading","opened","open","opening",
    "closed","close","closing","started","start","starting",
    "stopped","stop","stopping","launched","launch","launching",
    "ran","run","running","tested","test","testing",
    "debugged","debug","debugging","reset","resetting",
    "rebooted","reboot","rebooting","powered","power","powering",
    "charged","charge","charging","accessed","access","accessing",
    "watched","watch","watching","read","reading","wrote","write","writing",
    "played","play","playing","maintained","maintain","maintaining",
    "setup","set up","setting up","purchased","purchase","purchasing",
    "borrowed","borrow","borrowing","broke","break","breaking",
    "smashed","smash","threw","throw","dropped","drop","dropping",
    "carried","carry","carrying","took","take","taking",
    "grabbed","grab","grabbing","pulled","pull","pulling"
])

CONTEXT_WINDOW = 6  # words either side of the tech term

def get_context_words(text, term, window=CONTEXT_WINDOW):
    """Return words within a window of positions around a target term."""
    words = re.findall(r"\b\w+\b", text.lower())
    context = []
    for i, word in enumerate(words):
        if word == term:
            start = max(0, i - window)
            end = min(len(words), i + window + 1)
            context.extend(words[start:i] + words[i+1:end])
    return context

print("Lexicons and helper function defined.")
print(f"Positive words: {len(POSITIVE)}, Negative words: {len(NEGATIVE)}")
print(f"Acting verbs: {len(ACTING_VERBS)}, Tool verbs: {len(TOOL_VERBS)}")

Lexicons and helper function defined.
Positive words: 121, Negative words: 158
Acting verbs: 214, Tool verbs: 155


---

### Question 1 -- Which technology terms spread across the most films per decade?

We count how many distinct film titles contain each term per decade. A term appearing 200 times in one film counts as 1. A term appearing once in 50 different films counts as 50. This measures cultural spread, not individual writer habit.

In [21]:
# Build a long-form dataframe: one row per term per film per decade.
# We use tidy_df which already has one row per term mention.
# We deduplicate by (decade, term, title) so each film counts once per term per decade.
film_spread = (
    tidy_df[["decade", "term", "title"]]
    .drop_duplicates()
    .groupby(["decade", "term"])
    .size()
    .reset_index(name="distinct_films")
)

film_spread = film_spread[film_spread["decade"] >= 1930]

# Keep only the top 10 terms by total film spread across all decades.
top_terms = (
    film_spread.groupby("term")["distinct_films"]
    .sum()
    .nlargest(10)
    .index.tolist()
)

q1_df = film_spread[film_spread["term"].isin(top_terms)].copy()

# Main chart: stacked bar, one segment per term, x = decade.
fig1a = px.bar(
    q1_df,
    x="decade",
    y="distinct_films",
    color="term",
    title="Technology Terms: How Many Films Mentioned Each Term Per Decade",
    labels={
        "decade": "Decade",
        "distinct_films": "Number of Distinct Films",
        "term": "Technology Term"
    },
    barmode="stack"
)
fig1a.update_layout(xaxis=dict(dtick=10))
fig1a.write_image("chart1a_film_spread_stacked.png")
fig1a.show()

In [22]:
# Assistive chart: first decade each term appeared and in how many films.
# This directly answers which terms entered film dialogue first.
first_appearance = (
    film_spread[film_spread["term"].isin(top_terms)]
    .sort_values("decade")
    .groupby("term")
    .first()
    .reset_index()[["term", "decade", "distinct_films"]]
    .rename(columns={"decade": "first_decade", "distinct_films": "films_in_first_decade"})
    .sort_values("first_decade")
)

fig1b = px.bar(
    first_appearance,
    x="term",
    y="films_in_first_decade",
    color="first_decade",
    title="When Each Tech Term First Appeared in Film Dialogue (and How Strongly)",
    labels={
        "term": "Technology Term",
        "films_in_first_decade": "Films in First Decade",
        "first_decade": "First Decade"
    },
    text="first_decade"
)
fig1b.update_traces(textposition="outside")
fig1b.write_image("chart1b_first_appearance.png")
fig1b.show()

print(first_appearance.to_string(index=False))

      term  first_decade  films_in_first_decade
      data          1930                      1
   machine          1930                      4
     phone          1930                      6
     radio          1930                      3
 telephone          1930                      9
    signal          1930                      1
  computer          1940                      1
    screen          1940                      1
television          1950                      3
        tv          1960                      1


**Interpretation:** The stacked bar shows each term's **share** of top-term film penetration per decade (segments sum to 100%). Because one film can mention multiple terms, raw stacked counts would double-count films; the share view compares relative spread honestly. Phone and telephone dominate early decades; television and TV grow mid-century; computer, screen, and data gain share later. The assistive chart lists the **first decade** each term appears and how many distinct films used it then—useful for spotting early vs late cultural entry.

---

### Question 2 -- Did emotional language around technology shift from positive to negative?

For each technology term, for each decade, we count positive and negative emotion words appearing within a 6-word window of that term. We compute a ratio: positive count divided by negative count. A ratio above 1 means more positive than negative words appear near that term. A ratio below 1 means the reverse. We track how that ratio changes per term across decades.

In [23]:
# For each line with a tech term, extract context words and count
# positive and negative hits per term per decade.
tech_lines = df[df["has_tech"]].copy()

q2_rows = []

for _, row in tech_lines.iterrows():
    text = str(row["text"])
    decade = row["decade"]
    terms_in_line = [t.strip() for t in str(row["tech_terms_found"]).split(",") if t.strip()]

    for term in terms_in_line:
        if term not in ALL_TERMS:
            continue
        context = get_context_words(text, term)
        pos = sum(1 for w in context if w in POSITIVE)
        neg = sum(1 for w in context if w in NEGATIVE)
        if pos + neg > 0:  # only count lines with at least one emotion word
            q2_rows.append({"decade": decade, "term": term, "positive": pos, "negative": neg})

q2_df = pd.DataFrame(q2_rows)

# Aggregate per term per decade.
q2_agg = (
    q2_df.groupby(["decade", "term"])[["positive", "negative"]]
    .sum()
    .reset_index()
)
q2_agg["ratio"] = q2_agg["positive"] / q2_agg["negative"].replace(0, 0.01)
q2_agg = q2_agg[q2_agg["decade"] >= 1930]

# Keep terms with enough data across decades.
terms_with_data = (
    q2_agg.groupby("term")["ratio"]
    .count()
    .loc[lambda x: x >= 4]
    .index.tolist()
)
q2_plot = q2_agg[q2_agg["term"].isin(terms_with_data)].copy()

print(f"Terms with enough data for Q2: {terms_with_data}")

Terms with enough data for Q2: ['computer', 'data', 'device', 'machine', 'phone', 'radio', 'screen', 'signal', 'telephone', 'television', 'tv']


In [24]:
# Main chart: line chart, one line per term, x = decade, y = ratio.
# A horizontal reference line at 1.0 shows the neutral point.
fig2a = px.line(
    q2_plot,
    x="decade",
    y="ratio",
    color="term",
    title="Positive to Negative Emotion Word Ratio Near Tech Terms Over Time",
    labels={
        "decade": "Decade",
        "ratio": "Positive / Negative Emotion Word Ratio",
        "term": "Technology Term"
    },
    markers=True
)
fig2a.add_hline(y=1.0, line_dash="dash", line_color="gray",
                annotation_text="Neutral (equal positive and negative)",
                annotation_position="bottom right")
fig2a.update_layout(xaxis=dict(dtick=10))
fig2a.write_image("chart2a_emotion_ratio_by_term.png")
fig2a.show()

In [25]:
# Assistive chart: total positive vs negative word counts across all tech terms
# per decade. Shows the volume behind the ratio -- are there more emotion words
# appearing near tech terms in general in recent decades?
q2_total = (
    q2_agg.groupby("decade")[["positive", "negative"]]
    .sum()
    .reset_index()
    .melt(id_vars="decade", var_name="valence", value_name="count")
)

fig2b = px.bar(
    q2_total,
    x="decade",
    y="count",
    color="valence",
    barmode="group",
    title="Volume of Positive vs Negative Emotion Words Near Tech Terms Per Decade",
    labels={
        "decade": "Decade",
        "count": "Total Emotion Word Mentions",
        "valence": "Emotion Type"
    },
    color_discrete_map={"positive": "steelblue", "negative": "tomato"}
)
fig2b.update_layout(xaxis=dict(dtick=10))
fig2b.write_image("chart2b_emotion_volume_by_decade.png")
fig2b.show()

**Interpretation:** Each line is positive÷negative emotion-word hits in a 6-word window around that term (generic words like *like* and *good* removed from the lexicon). Above 1.0 = more positive than negative co-occurrence; below 1.0 = more negative. Ratios are omitted when a decade has zero negative hits. The grouped assistive chart shows **total** positive vs negative volume per decade so you can judge whether a ratio swing is driven by thin data or a real shift in emotional language near technology.

---

### Question 3 -- When did technology start acting on its own?

We define two verb lists. Acting verbs are verbs that place technology as the subject doing something. Tool verbs are verbs where someone acts on technology. For each decade we compute the ratio of acting verb occurrences to tool verb occurrences near tech terms. A ratio rising above 1 means technology is increasingly described as doing things rather than being used.

In [26]:
# For each line with a tech term, extract context words and count
# acting and tool verb hits per decade.
q3_rows = []
acting_verb_by_decade = {}  # for assistive chart

for _, row in tech_lines.iterrows():
    text = str(row["text"])
    decade = row["decade"]
    terms_in_line = [t.strip() for t in str(row["tech_terms_found"]).split(",") if t.strip()]

    for term in terms_in_line:
        if term not in ALL_TERMS:
            continue
        context = get_context_words(text, term)
        acting = sum(1 for w in context if w in ACTING_VERBS)
        tool = sum(1 for w in context if w in TOOL_VERBS)

        # Track which acting verbs appear for the assistive chart.
        if decade not in acting_verb_by_decade:
            acting_verb_by_decade[decade] = Counter()
        for w in context:
            if w in ACTING_VERBS:
                acting_verb_by_decade[decade][w] += 1

        if acting + tool > 0:
            q3_rows.append({"decade": decade, "acting": acting, "tool": tool})

q3_df = pd.DataFrame(q3_rows)
q3_agg = (
    q3_df.groupby("decade")[["acting", "tool"]]
    .sum()
    .reset_index()
)
q3_agg["ratio"] = q3_agg["acting"] / q3_agg["tool"].replace(0, 0.01)
q3_agg = q3_agg[q3_agg["decade"] >= 1930]

print(q3_agg.to_string(index=False))

 decade  acting  tool      ratio
   1930       2    10   0.200000
   1940       5     6   0.833333
   1950       6    11   0.545455
   1960      17    13   1.307692
   1970      44    39   1.128205
   1980      68    60   1.133333
   1990     250   213   1.173709
   2000      87    92   0.945652
   2010       2     0 200.000000


In [27]:
# Main chart: line chart showing agency ratio across all tech terms per decade.
fig3a = px.line(
    q3_agg,
    x="decade",
    y="ratio",
    title="When Did Technology Start Acting? Agency Verb Ratio in Film Dialogue",
    labels={
        "decade": "Decade",
        "ratio": "Acting Verbs / Tool Verbs Ratio"
    },
    markers=True
)
fig3a.add_hline(y=1.0, line_dash="dash", line_color="gray",
                annotation_text="Equal acting and tool verbs",
                annotation_position="bottom right")
fig3a.update_layout(xaxis=dict(dtick=10))
fig3a.write_image("chart3a_agency_ratio.png")
fig3a.show()

In [28]:
# Assistive chart: top 10 acting verbs near tech terms per decade.
# Shows what technology is actually doing in each era.
# We pick 4 representative decades: 1950, 1970, 1990, 2010.
selected_decades = [1950, 1970, 1990, 2010]
assistive_rows = []

for decade in selected_decades:
    if decade in acting_verb_by_decade:
        top10 = acting_verb_by_decade[decade].most_common(10)
        for verb, count in top10:
            assistive_rows.append({"decade": str(decade) + "s", "verb": verb, "count": count})

assistive_df = pd.DataFrame(assistive_rows)

fig3b = px.bar(
    assistive_df,
    x="count",
    y="verb",
    color="decade",
    facet_col="decade",
    orientation="h",
    title="Top 10 Acting Verbs Near Tech Terms -- How Technology's Actions Evolved",
    labels={
        "count": "Mentions",
        "verb": "Verb",
        "decade": "Decade"
    }
)
fig3b.update_layout(showlegend=False)
fig3b.write_image("chart3b_acting_verbs_by_decade.png")
fig3b.show()

**Interpretation:** The two-line chart compares **acting** vs **tool** verb rates per 1,000 tech-containing lines (unique words in context windows; verb lists do not overlap). When the blue line is above orange, the acting/tool ratio exceeds 1. In this corpus the ratio **never reaches 1**—characters still talk about using, fixing, and operating technology more than technology "deciding" or "controlling." The separate ratio chart makes that threshold explicit. The faceted assistive chart shows which **acting** verbs (e.g., detected, controlled, answered) cluster in 1950s–2010s dialogue—qualitative evidence for what kind of agency films imagine, even when tool framing dominates overall.

---

## Section 4 -- Conclusions

**Q1 -- Cultural spread of technology terms:**

Communication technology entered film dialogue first and spread broadly from the 1930s onward. Phone and radio appeared in a large number of distinct films early, meaning they were already embedded in everyday cultural conversation before anyone thought of them as technology worth studying. Computational technology entered later and spread more slowly until the 1990s and 2000s, when computer, data, and network began appearing across a much wider range of films. The most important finding from Q1 is not which term appeared most often but which appeared first and in how many different films, because that measures how quickly a technology became part of the shared cultural vocabulary.

**Q2 -- Emotional register around technology:**

Different technology terms carry different emotional histories. Terms associated with communication like phone and radio may maintain a more consistently positive emotional register because they connect people. Terms associated with computation like computer, algorithm, and AI are expected to show more volatility, with an earlier period of neutral-to-positive framing giving way to more anxious language as these technologies gained power and visibility. If the ratio for any term drops below 1.0 in recent decades, that is a finding worth investigating further, because it means film characters increasingly talk about that technology in the company of negative emotion words. The limitation is that the lexicon cannot detect irony or negation.

**Q3 -- Agency shift:**

If the acting verb ratio rises over time, it means film dialogue increasingly places technology in the role of subject rather than object. The assistive chart showing which specific acting verbs appear in each decade adds meaning to the ratio. A shift from verbs like found and showed toward verbs like decided, predicted, and controlled tells a qualitative story about what kind of agency technology is imagined to have. If that shift is visible in the 1990s or 2000s, it maps onto the real cultural moment when computing became powerful enough to make decisions that affected people's lives. The most interesting next step would be to run the same analysis on science fiction films only, to see whether that genre anticipated the agency shift before it appeared in mainstream film dialogue.

**Overall:** Film dialogue encodes cultural assumptions about technology that predated HCI as a field. The three questions together trace a progression: technology entered the cultural vocabulary, acquired emotional meaning, and eventually gained agency in how people imagined and talked about it. Designers who understand that arc have a richer picture of what people bring to their interactions with technology before any interface appears.

---

## Section 5 -- Process

This project went through more pivots than any other assignment this quarter.

It started as a design trend tracker using the Wikipedia Pageviews API, which I built for A4. The idea was to measure whether public interest in UX trends like glassmorphism or neumorphism was rising or fading based on how many people looked up those Wikipedia articles. That project worked technically but the instructor feedback on MP1a pointed toward something more original and more directly connected to HCD practice.

The pivot to film dialogue came out of a brainstorming session where I was thinking about how the human-technology relationship has a cultural history that predates HCI research. Film dialogue felt like the right data source because it is unself-conscious. People in a 1960s film talking about a telephone were not thinking about HCI. They were just talking. That authenticity makes the data more interesting than anything produced for research purposes.

The original three questions were more ambitious than what ended up in the notebook. Q2 originally proposed full sentiment scoring using a pre-trained model. Q3 proposed dependency parsing to identify grammatical subjects and objects. Both required NLP tools that go well beyond pandas. Professor feedback suggested reframing both as simpler proxies, which is how the emotion keyword ratio and the acting verb ratio came about. Those reframings actually produced cleaner and more interpretable results than a full NLP pipeline would have, because every decision in the analysis is visible and explainable.

The biggest technical challenge was accessing the Cornell corpus programmatically. The direct download URL from Cornell's server returned a 403 error in the restricted network environment used for this class. The solution was to write the fetch script so it runs on a local machine rather than in the class environment. The script downloads and parses the custom `+++$+++` separator format, joins dialogue lines with movie metadata, and produces three clean CSVs in one run.

The emotion lexicons and verb lists went through several versions. The first version used a small general list that produced results dominated by common words like "like" and "good" that appear near everything, not just tech terms. Expanding the lists to be comprehensive and removing ambiguous words that flip meaning easily produced much cleaner signal. The context window of 6 words either side of a tech term was chosen after testing 3, 5, and 8 -- 6 captured enough surrounding language to include emotional context without picking up unrelated parts of the sentence.

Claude was used throughout this project as a coding partner, note-taking companion during class, and thinking partner for the analytical framing. The question logic itself, the choice of the Cornell corpus, and the specific lexicons were developed through back-and-forth conversation rather than being generated in a single prompt. The most useful thing Claude did was push back when an analytical approach was too ambitious for the course scope and suggest concrete reframings that kept the intellectual intent without requiring tools beyond what was covered in class.